<h1> PART 1: ANY MATHMAITCAL QUESTIONS </h1>

# THE FINAL SUBMISSION FILE IS CREATED AS final_submission.csv in output.  

### important:- the rag kb indexed is the dataset tittled "rag-corpus-final-but-without-bm25". this was pinned to version 1 and should be kept at version 1.
#### The notebook should finish in sub 5hr for 5k samples

In [ ]:
import subprocess
import sys

# 1. Clear out heavy frameworks that are known to cause dependency deadlocks
print("Uninstalling conflicting baseline packages...")
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y', 
    'keras', 'tensorflow', 'tensorboard'
], check=True)

# 2. Install vllm using the specific CUDA 12 wheel, along with its ecosystem
print("Installing vllm ecosystem (this may take a few minutes)...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install', 
    '--upgrade', 
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    '--extra-index-url', 'https://pypi.nvidia.com',
    # Bypass PyPI and use the official CUDA 12.9 wheel for vLLM 0.24.0
    'https://github.com/vllm-project/vllm/releases/download/v0.24.0/vllm-0.24.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl', 
    'unsloth', 
    'trl', 
    'openai_harmony'
], check=True)

print("Installation complete! You can now import your packages.")

In [ ]:
import re
import os
import gc
import math
import time
import traceback
import threading
import multiprocessing
from fractions import Fraction
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List, Tuple

import pandas as pd
import numpy as np

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# vLLM imports
from vllm import LLM, SamplingParams


In [ ]:
# ==========================================================================
# 1. Bengali <-> numeric heuristic utilities
# ==========================================================================

_BN_DIGITS = "০১২৩৪৫৬৭৮৯"
_BN_TO_EN_DIGIT_TABLE = str.maketrans(_BN_DIGITS, "0123456789")

def bn_digits_to_en(text: str) -> str:
    """Convert Bengali digit characters in text to ASCII digits."""
    return text.translate(_BN_TO_EN_DIGIT_TABLE)

# Bengali number words 0-99 (includes common spelling variants).
BN_NUM_WORDS = {
    "শূন্য": 0,
    "এক": 1, "দুই": 2, "দু": 2, "তিন": 3, "চার": 4, "পাঁচ": 5, "পাচ": 5,
    "ছয়": 6, "সাত": 7, "আট": 8, "নয়": 9,
    "দশ": 10, "এগারো": 11, "এগার": 11, "বারো": 12, "বার": 12, "তেরো": 13, "তের": 13,
    "চৌদ্দ": 14, "পনেরো": 15, "পনের": 15, "ষোল": 16, "ষোলো": 16, "সতেরো": 17, "সতের": 17,
    "আঠারো": 18, "আঠার": 18, "উনিশ": 19,
    "বিশ": 20, "একুশ": 21, "বাইশ": 22, "তেইশ": 23, "চব্বিশ": 24, "পঁয়ত্রিশ": 25, "পঁচিশ": 25, "পচিশ": 25,
    "ছাব্বিশ": 26, "সাতাশ": 27, "আটাশ": 28, "ঊনত্রিশ": 29, "উনত্রিশ": 29,
    "ত্রিশ": 30, "একত্রিশ": 31, "বত্রিশ": 32, "তেত্রিশ": 33, "চৌত্রিশ": 34,
    "পঁয়ত্রিশ": 35, "পয়ত্রিশ": 35, "ছত্রিশ": 36, "সাঁইত্রিশ": 37, "সাইত্রিশ": 37,
    "আটত্রিশ": 38, "ঊনচল্লিশ": 39, "উনচল্লিশ": 39,
    "চল্লিশ": 40, "একচল্লিশ": 41, "বিয়াল্লিশ": 42, "বেয়াল্লিশ": 42, "তেতাল্লিশ": 43,
    "চুয়াল্লিশ": 44, "পঁয়তাল্লিশ": 45, "পয়তাল্লিশ": 45, "ছেচল্লিশ": 46, "সাতচল্লিশ": 47,
    "আটচল্লিশ": 48, "ঊনপঞ্চাশ": 49, "উনপঞ্চাশ": 49,
    "পঞ্চাশ": 50, "একান্ন": 51, "বাহান্ন": 52, "তেপ্পান্ন": 53, "তিপ্পান্ন": 53,
    "চুয়ান্ন": 54, "পঞ্চান্ন": 55, "ছাপ্পান্ন": 56, "সাতান্ন": 57, "আটান্ন": 58,
    "ঊনষাট": 59, "উনষাট": 59,
    "ষাট": 60, "একষট্টি": 61, "বাষট্টি": 62, "তেষট্টি": 63, "চৌষট্টি": 64,
    "পঁয়ষট্টি": 65, "পয়ষট্টি": 65, "ছেষট্টি": 66, "সাতষট্টি": 67, "আটষট্টি": 68,
    "ঊনসত্তর": 69, "উনসত্তর": 69,
    "সত্তর": 70, "একাত্তর": 71, "বাহাত্তর": 72, "তিয়াত্তর": 73, "তেহাত্তর": 73,
    "চুয়াত্তর": 74, "পঁচাত্তর": 75, "পচাত্তর": 75, "ছিয়াত্তর": 76, "সাতাত্তর": 77,
    "আটাত্তর": 78, "ঊনআশি": 79, "উনআশি": 79,
    "আশি": 80, "একাশি": 81, "বিরাশি": 82, "তিরাশি": 83, "চুরাশি": 84, "চৌরাশি": 84,
    "পঁচাশি": 85, "পচাশি": 85, "ছিয়াশি": 86, "সাতাশি": 87, "আটাশি": 88,
    "ঊননব্বই": 89, "উননব্বই": 89,
    "নব্বই": 90, "একানব্বই": 91, "বিরানব্বই": 92, "তিরানব্বই": 93, "চুরানব্বই": 94,
    "পঁচানব্বই": 95, "পচানব্বই": 95, "ছিয়ানব্বই": 96, "সাতানব্বই": 97, "আটানব্বই": 98,
    "নিরানব্বই": 99,
}

# Multiplier / scale words.
BN_SCALE_WORDS = {
    "শত": 100, "শো": 100, "শ": 100,
    "হজার": 1000, "হাজার": 1000,
    "লক্ষ": 100000, "লাখ": 100000,
    "নিযুত": 1000000,
    "কোটি": 10000000,
}

DECIMAL_WORD = "দশমিক"

def _tokenize_bn(text: str):
    return re.findall(r"[\u0980-\u09FF]+", text)

def _resolve_token_value(tok: str):
    if tok in BN_NUM_WORDS:
        return ("digit", BN_NUM_WORDS[tok])
    if tok in BN_SCALE_WORDS:
        return ("scale", BN_SCALE_WORDS[tok])
    for suffix in ("শত", "শো", "শ"):
        if tok.endswith(suffix) and len(tok) > len(suffix):
            root = tok[: -len(suffix)]
            if root in BN_NUM_WORDS and 1 <= BN_NUM_WORDS[root] <= 9:
                return ("digit", BN_NUM_WORDS[root] * 100)
    if tok in ("একশ", "একশো", "একশত"):
        return ("digit", 100)
    return None

def parse_bn_number_words(text: str):
    tokens = _tokenize_bn(text)
    results = []
    current, total, have_value = 0, 0, False
    for tok in tokens:
        if tok == DECIMAL_WORD:
            continue
        resolved = _resolve_token_value(tok)
        if resolved is None:
            if have_value:
                results.append(total + current)
                current, total, have_value = 0, 0, False
        elif resolved[0] == "digit":
            current += resolved[1]
            have_value = True
        else:
            mult = resolved[1]
            current = current or 1
            total += current * mult
            current = 0
            have_value = True
    if have_value:
        results.append(total + current)
    return results

def extract_numeric_candidates(text: str):
    converted = bn_digits_to_en(text)
    converted = re.sub(r"(\d)\s*দশমিক\s*(\d)", r"\1.\2", converted)

    candidates = [float(m.group(0)) for m in re.finditer(r"-?\d+(?:\.\d+)?", converted)]
    if not candidates:
        candidates = [float(v) for v in parse_bn_number_words(text)]
    return candidates

def values_match(calculated, candidates, rel_tol=1e-3, abs_tol=1e-2):
    if calculated is None:
        return None
    try:
        calc = float(calculated)
    except (TypeError, ValueError):
        return None
    if not candidates:
        return None
    for c in candidates:
        if abs(c - calc) <= max(abs_tol, rel_tol * max(abs(c), abs(calc))):
            return True
    return False

def is_math_problem(prompt_bn):
    text = prompt_bn.strip()
    if not text:
        return False

    math_keywords = {
        'অনুপাত', 'গড়', 'দিন', 'ঘণ্টা', 'মিনিট', 'সেকেন্ড', 'টাকা', 'পয়সা',
        'শতকরা', 'লাভ', 'ক্ষতি', 'সুদ', 'মূলধন', 'সুদ-আসল', 'চক্রবৃদ্ধি', 'সরল সুদ',
        'বয়স', 'দূরত্ব', 'বেগ', 'সময়', 'গতিবেগ', 'কার্য', 'শ্রম', 'পাইপ', 'চৌবাচ্চা',
        'বর্গ', 'ক্ষেত্রফল', 'পরিসীমা', 'আয়তন', 'ঘনফল', 'ত্রিভুজ', 'বৃত্ত', 'কোণ',
        'ডিগ্রি', 'রেডিয়ান', 'লগারিদম', 'সূচক', 'গুণোত্তর', 'সমান্তর', 'ধারা',
        'সংখ্যা', 'পূর্ণসংখ্যা', 'ভগ্নাংশ', 'দশমিক', 'পরিমাপ', 'ওজন', 'ভর',
        'তাপমাত্রা', 'বল', 'শক্তি', 'কাজ', 'ক্ষমতা', 'চাপ', 'প্রতিরোধ', 'রোধ',
        'তড়িৎ', 'ভোল্ট', 'অ্যাম্পিয়ার', 'ওয়াট', 'হার', 'ট্যাক্স', 'কর', 'মুনাফা',
        'বিনিয়োগ', 'ঋণ', 'কিস্তি', 'বাট্টা', 'ছাড়', 'মূল্য', 'ক্রয়', 'বিক্রয়',
        'দ্রব্য', 'পণ্য', 'পরিমাণ', 'মোট', 'যোগফল', 'বিয়োগফল', 'গুণফল', 'ভাগফল',
        'গসাগু', 'লসাগু', 'মৌলিক', 'যৌগিক', 'পূর্ণবর্গ', 'বর্গমূল', 'ঘনমূল',
        'সমীকরণ', 'অসমতা', 'চলক', 'ধ্রুবক', 'প্রতিসম', 'অনুক্রম', 'সেট', 'উপাদান',
        'সম্ভাবনা', 'পরিসংখ্যান', 'মধ্যমা', 'প্রচুরক', 'বিচ্যুতি', 'বিন্যাস', 'সমাবেশ',
        'হেক্সাডেসিমেল', 'অক্টাল', 'বাইনারী', 'ভিত্তি', 'বেস',
        'অক্টালে', 'বাইনারি', 'হেক্সা', 'সংখ্যা পদ্ধতি',
        'রূপান্তর', 'কনভার্ট', 'পরিবর্তন'
    }

    math_units = {
        'মিটার', 'কিলোমিটার', 'সেন্টিমিটার', 'মিলিমিটার', 'ফুট', 'ইঞ্চি', 'গজ',
        'লিটার', 'মিলিলিটার', 'কিউবিক', 'বর্গ', 'কেজি', 'গ্রাম', 'পাউন্ড', 'টন',
        'কিলোগ্রাম', 'মিলিগ্রাম', 'ঘন্টা', 'মিনিট', 'সেকেন্ড', 'দিন', 'সপ্তাহ', 'মাস', 'বছর',
        'ডিগ্রি', 'রেডিয়ান', 'পিপিএম', 'পারসেন্ট', 'শতাংশ'
    }

    math_ops = {
        'যোগ', 'বিয়োগ', 'গুণ', 'ভাগ', 'হার', 'গড়', 'মোট', 'যোগফল', 'অন্তর',
        'গুণফল', 'ভাগফল', 'বর্গ', 'বর্গমূল', 'ঘন', 'ঘনমূল', 'লগ', 'সূচক',
        'সমাধান', 'নির্ণয়', 'হিসাব', 'পরিমাণ', 'পরিমাপ', 'গণনা', 'অঙ্ক'
    }

    math_frames = {
        'একা', 'একত্রে', 'যৌথভাবে', 'প্রত্যেক', 'প্রতি', 'মোট', 'অবশিষ্ট', 'বাকি',
        'যদি', 'তবে', 'হলে', 'হয়', 'নয়', 'সমান', 'বড়', 'ছোট', 'অಧಿಕ', 'কম',
        'অর্ধেক', 'দ্বিগুণ', 'তিনগুণ', 'চারগুণ', 'পাঁচগুণ', 'দশগুণ', 'শতগুণ'
    }

    digit_pattern = re.compile(r'[০-৯0-9]')
    ratio_pattern = re.compile(r'[০-৯0-9]+\s*[:/]\s*[০-৯0-9]+')
    percent_pattern = re.compile(r'[০-৯0-9]+\s*%')
    sqrt_pattern = re.compile(r'√')
    fraction_pattern = re.compile(r'[০-৯0-9]+/[০-৯0-9]+|½|¼|¾|⅓|⅔|⅕|⅖|⅗|⅘|⅙|⅚|⅛|⅜|⅝|⅞')
    eq_pattern = re.compile(r'=|সমান')
    base_pattern = re.compile(r'\([0-9A-Fa-f০-৯]+\)\s*[০-৯0-9]+')

    score = 0
    if digit_pattern.search(text):
        score += 2

    words = set(re.findall(r'[\u0980-\u09FF]+', text))
    if any(kw in words for kw in math_keywords):
        score += 3
    if any(kw in words for kw in math_units):
        score += 2
    if any(kw in words for kw in math_ops):
        score += 2
    if any(kw in words for kw in math_frames):
        score += 1

    if ratio_pattern.search(text):
        score += 3
    if percent_pattern.search(text):
        score += 3
    if sqrt_pattern.search(text):
        score += 3
    if fraction_pattern.search(text):
        score += 3
    if eq_pattern.search(text):
        score += 2
    if base_pattern.search(text):
        score += 3

    if 'কত' in text:
        score += 1

    non_math_date_keywords = {'সাল', 'সালে', 'খ্রীষ্টাব্দ', 'ইং', 'তারিখ', 'তারিখে', 'জন্ম', 'জন্মগ্রহণ', 'মৃত্যু', 'মৃত্যুবরণ', 'প্রতিষ্ঠা', 'উদ্বোধন', 'কবে', 'কোথায়', 'কীভাবে'}
    for kw in non_math_date_keywords:
        if kw in words or kw in text:
            score -= 5

    return score >= 5

def split_thinking(llm_output: str):
    if "</think>" in llm_output:
        idx = llm_output.rfind("</think>")
        return llm_output[:idx], llm_output[idx + len("</think>"):]
    return "", llm_output

_CODE_FENCE_RE = re.compile(r'```(?:python)?\s*\n?(.*?)\s*```', re.DOTALL | re.IGNORECASE)
_OPEN_FENCE_RE = re.compile(r'```(?:python)?\s*\n?(.*)', re.DOTALL | re.IGNORECASE)

def _non_empty_blocks(text):
    return [b.strip() for b in _CODE_FENCE_RE.findall(text) if b.strip()]

def extract_python_code(llm_output):
    _, final_part = split_thinking(llm_output)

    blocks = _non_empty_blocks(final_part)
    if blocks:
        for b in reversed(blocks):
            if "final_answer" in b:
                return b
        return blocks[-1]

    m = _OPEN_FENCE_RE.search(final_part)
    if m and m.group(1).strip():
        return m.group(1).strip()

    blocks = _non_empty_blocks(llm_output)
    if blocks:
        for b in reversed(blocks):
            if "final_answer" in b:
                return b
        return blocks[-1]

    return final_part.strip() or llm_output.strip()

_VERDICT_RE = re.compile(r'VERDICT\s*:\s*([01])', re.IGNORECASE)

def parse_verdict(llm_output):
    thinking, final = split_thinking(llm_output)

    m = _VERDICT_RE.search(final)
    if m:
        return int(m.group(1)), thinking.strip()

    stripped = final.strip()
    if stripped in ("0", "1"):
        return int(stripped), thinking.strip()

    m2 = re.match(r'\s*([01])\b', stripped)
    if m2:
        return int(m2.group(1)), thinking.strip()

    return None, thinking.strip()

def safe_execute_math(python_code: str):
    import math as _math
    import numpy as _np
    from fractions import Fraction as _Fraction
    try:
        import sympy as _sympy
    except ImportError:
        _sympy = None

    local_vars = {}
    safe_builtins = {
        "abs": abs, "round": round, "min": min, "max": max, "sum": sum,
        "len": len, "int": int, "float": float, "pow": pow, "divmod": divmod,
        "range": range, "enumerate": enumerate, "zip": zip, "map": map,
        "sorted": sorted, "reversed": reversed, "list": list, "tuple": tuple,
        "set": set, "dict": dict, "str": str, "bool": bool,
        "print": print,
        "oct": oct, "hex": hex, "bin": bin,
        "__import__": __import__,
    }
    safe_globals = {
        "__builtins__": safe_builtins,
        "math": _math,
        "np": _np,
        "numpy": _np,
        "Fraction": _Fraction,
    }
    if _sympy is not None:
        safe_globals["sympy"] = _sympy

    try:
        exec(python_code, safe_globals, local_vars)
        if 'final_answer' in local_vars:
            return local_vars['final_answer']
    except Exception as e:
        # Print helper for sandbox debug
        pass
    return None

def _run_with_timeout(code: str, timeout: int = 5) -> Optional[float]:
    """Runs safe_execute_math inside a separate process to prevent infinite loops from hanging."""
    def worker(code, queue):
        try:
            res = safe_execute_math(code)
            queue.put(res)
        except Exception:
            queue.put(None)

    queue = multiprocessing.Queue()
    proc = multiprocessing.Process(target=worker, args=(code, queue))
    proc.start()
    proc.join(timeout)
    if proc.is_alive():
        print(f"[safe_exec] Timeout after {timeout}s. Terminating process.")
        proc.terminate()
        proc.join()
        return None
    if not queue.empty():
        return queue.get()
    return None


In [ ]:
class CFG:
    model_name = "Qwen/Qwen3-8B"
    
    # SC-TIR configs
    N_SAMPLES = 1
    temperature = 0.1
    top_p = 0.95
    max_tokens = 2048
    
    # vLLM Configs
    gpu_memory_utilization = 0.96
    dtype = "float16"
    max_model_len = 8192
    
    # Voting Tolerance
    numeric_tol = 1e-2
    
    # Paths
    output_csv_path = "predictions.csv"

def find_test_csv() -> str:
    """Robust helper to find 'test set.csv' recursively in kaggle inputs or locally."""
    paths_to_try = [

        "/kaggle/input/competitions/bengali-hallucination/test set.csv"
        # "/kaggle/input/competitions/bengali-hallucination/test set.csv",
        # "/kaggle/input/iut-datathon/test set.csv",
        # "/kaggle/input/test set.csv",
        # "Helpers/test set.csv",
        # "../Helpers/test set.csv",
        # "test set.csv"
    ]
    for p in paths_to_try:
        if os.path.exists(p):
            return p
            
    # Recursive search under /kaggle/input
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith("test set.csv"):
                    return os.path.join(root, f)
                    
    raise FileNotFoundError("test set.csv could not be found anywhere!")

def load_vllm_engine(cfg: CFG) -> LLM:
    print(f"[vLLM] Initializing LLM Engine with: {cfg.model_name}")
    llm = LLM(
        model=cfg.model_name,
        dtype=cfg.dtype,
        gpu_memory_utilization=cfg.gpu_memory_utilization,
        tensor_parallel_size=2,
        max_model_len=cfg.max_model_len,
        trust_remote_code=True,
        enforce_eager=True
    )
    print("[vLLM] Engine Loaded Successfully!")
    return llm


In [ ]:
SC_SYSTEM_PROMPT = (
    "You are an expert mathematician and Python programmer. "
    "Your task is to solve Bengali math word problems by writing Python code.\n\n"
    "# Guidelines\n"
    "1. Read the Bengali math problem carefully. Translate numbers and words correctly.\n"
    "2. Think step by step.\n"
    "3. You can import packages like `math`, `fractions`, `numpy`, and `sympy` to help solve the problem.\n"
    "4. Save the final calculated answer (must be a single int or float) to a variable named `final_answer`.\n"
    "5. Produce only the python code block inside a code block: ```python ... ```. Do not provide prose explanations."
)

SC_USER_TEMPLATE = (
    "Solve the following Bengali math word problem by writing Python code:\n\n"
    "Problem: {problem}\n\n"
    "Write the Python code to compute `final_answer`.\n"
    "Output only the code block, no explanations."
)

def build_sc_prompt(problem: str, tokenizer) -> str:
    messages = [
        {"role": "system", "content": SC_SYSTEM_PROMPT},
        {"role": "user", "content": SC_USER_TEMPLATE.format(problem=problem)}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_all_samples(math_problems: List[str], llm: LLM, tokenizer, cfg: CFG) -> List[List[str]]:
    P = len(math_problems)
    N = cfg.N_SAMPLES
    
    all_prompts = []
    for prob in math_problems:
        prompt = build_sc_prompt(prob, tokenizer)
        for _ in range(N):
            all_prompts.append(prompt)
            
    sampling_params = SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=cfg.max_tokens
    )
    
    print(f"[vLLM] Running generation of {len(all_prompts)} total samples ({P} problems x {N} samples each)...")
    t0 = time.time()
    outputs = llm.generate(all_prompts, sampling_params)
    print(f"[vLLM] Generation finished in {time.time()-t0:.2f} seconds.")
    
    results = []
    for i in range(P):
        problem_samples = []
        for j in range(N):
            idx = i * N + j
            problem_samples.append(outputs[idx].outputs[0].text)
        results.append(problem_samples)
    return results

def execute_samples_parallel(per_problem_raw: List[List[str]], max_workers: int = 16) -> List[List[Optional[float]]]:
    P = len(per_problem_raw)
    N = len(per_problem_raw[0]) if P > 0 else 0
    
    tasks = []
    for i, samples in enumerate(per_problem_raw):
        for j, text in enumerate(samples):
            tasks.append((i, j, text))

    flat_answers = {}
    lock = threading.Lock()

    def run_one(task):
        i, j, raw_text = task
        code = extract_python_code(raw_text)
        ans = _run_with_timeout(code, timeout=5)
        with lock:
            flat_answers[(i, j)] = ans

    print(f"[Executor] Running execution of {len(tasks)} code buffers in parallel...")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(run_one, t) for t in tasks]
        for fut in as_completed(futures):
            pass
    print(f"[Executor] All code buffers executed in {time.time()-t0:.2f} seconds.")
    
    results = []
    for i in range(P):
        problem_ans = []
        for j in range(N):
            problem_ans.append(flat_answers.get((i, j), None))
        results.append(problem_ans)
    return results

def numeric_vote(answers: List[Optional[float]], tol: float = 1e-2) -> Optional[float]:
    valid = []
    for val in answers:
        if val is None:
            continue
        try:
            valid.append(float(val))
        except (TypeError, ValueError):
            continue

    if not valid:
        return None
    if len(valid) == 1:
        return valid[0]
        
    clusters = []
    for v in valid:
        placed = False
        for cluster in clusters:
            rep = sum(cluster) / len(cluster)
            if abs(v - rep) <= max(tol, tol * abs(rep)):
                cluster.append(v)
                placed = True
                break
        if not placed:
            clusters.append([v])
            
    best = max(clusters, key=lambda c: (len(c), -abs(sum(c)/len(c))))
    return round(sum(best) / len(best), 10)


In [ ]:
def run_sctir_math_pipeline(df: pd.DataFrame, llm: LLM, tokenizer, cfg: CFG) -> pd.DataFrame:
    df = df.copy()
    df["predicted_label"] = None
    df["calculated_answer"] = None
    df["judge_source"] = None
    df["judge_rationale"] = None
    df["sc_vote_info"] = None
    
    math_mask = df["prompt_bn"].apply(is_math_problem)
    math_indices = df.index[math_mask].tolist()
    math_problems = df.loc[math_indices, "prompt_bn"].tolist()
    
    df.loc[~math_mask, "judge_source"] = "non_math"
    
    if not math_problems:
        print("[Pipeline] No math problems found.")
        return df
        
    print(f"[Pipeline] Found {len(math_problems)} math problems out of {len(df)} total rows.")
    
    # Step 1: Generate samples
    per_problem_raw = generate_all_samples(math_problems, llm, tokenizer, cfg)
    
    # Step 2: Parallel execute samples
    per_problem_answers = execute_samples_parallel(per_problem_raw, max_workers=min(cfg.N_SAMPLES * 4, 32))
    
    # Step 3: Voting & Heuristic checking
    comparator_prompts = []
    comparator_indices = []
    
    for idx_local, (idx, answers) in enumerate(zip(math_indices, per_problem_answers)):
        elected = numeric_vote(answers, tol=cfg.numeric_tol)
        df.at[idx, "calculated_answer"] = elected
        df.at[idx, "sc_vote_info"] = str({
            "answers": answers,
            "elected": elected
        })
        
        if elected is None:
            df.at[idx, "judge_source"] = "exec_fail"
            continue
            
        response_bn = df.iloc[idx]["response_bn"]
        candidates = extract_numeric_candidates(str(response_bn))
        h_result = values_match(elected, candidates)
        
        if h_result is not None:
            df.at[idx, "predicted_label"] = int(h_result)
            df.at[idx, "judge_source"] = "heuristic"
            df.at[idx, "judge_rationale"] = f"calc={elected} vs candidates={candidates} -> {'match' if h_result else 'mismatch'}"
            continue
            
        # fallback to LLM
        comp_text = (
            f"You are an absolute numerical verification judge.\n"
            f"Calculated true numerical value: {elected}\n"
            f"Provided response text: {response_bn}\n\n"
            f"Task: Check if the provided response text contains the exact same numerical value/meaning as the calculated true value.\n"
            f"- You must perfectly understand Bengali number words (e.g. 'একাত্তর' means 71, 'পাঁচ' means 5).\n"
            f"- Ignore trailing words or measurement units like (দিন, টাকা, বছর, জন, টি).\n"
            f"- If the fraction or decimal values match exactly, consider it a match.\n\n"
            f"After you finish reasoning, output your final answer as exactly one line in this format and nothing else on that line:\n"
            f"VERDICT: 1\n"
            f"or\n"
            f"VERDICT: 0"
        )
        messages = [{"role": "user", "content": comp_text}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        comparator_prompts.append(formatted)
        comparator_indices.append(idx)
        
    if comparator_prompts:
        print(f"\n[Pipeline] Running LLM Judge Fallback on {len(comparator_prompts)} unresolved rows...")
        comp_params = SamplingParams(temperature=0.0, max_tokens=1024)
        comp_outputs = llm.generate(comparator_prompts, comp_params)
        
        for k, comp_out in enumerate(comp_outputs):
            raw_decision = comp_out.outputs[0].text
            idx = comparator_indices[k]
            verdict, thinking = parse_verdict(raw_decision)
            
            df.at[idx, "predicted_label"] = verdict
            df.at[idx, "judge_source"] = "llm" if verdict is not None else "unresolved"
            df.at[idx, "judge_rationale"] = thinking
            if verdict is None:
                print(f"[WARNING] Failed to parse verdict for row {idx}")
                
    print("[Pipeline] SC-TIR Math Pipeline Execution Finished.")
    return df


In [ ]:
if __name__ == "__main__":
    cfg = CFG()
    
    # 1. Find and load the test set
    csv_path = find_test_csv()
    print(f"[Main] Loading test set from: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"[Main] Loaded {len(df)} rows.")
    
    # Optional: subset for test
    # df = df.head(10)
    
    # 2. Tokenizer initialization
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, padding_side="left", trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    # 3. Load vLLM engine
    llm = load_vllm_engine(cfg)
    
    # 4. Run pipeline
    df_predictions = run_sctir_math_pipeline(df, llm, tokenizer, cfg)
    
    # 5. Save output
    df_predictions.to_csv(cfg.output_csv_path, index=False)
    print(f"[Main] Saved predictions to: {cfg.output_csv_path}")

    # 6. Save the rest of the dataset for the next pipeline stage
    # Only keep rows where judge_source != "heuristic"
    rest_df = df_predictions[df_predictions["judge_source"] != "heuristic"].copy()
    # Keep only original columns from the test set
    original_cols = df.columns.tolist()
    rest_df = rest_df[original_cols]
    
    rest_csv_path = "rest_test_set.csv" 
    rest_df.to_csv(rest_csv_path, index=False)
    print(f"[Main] Saved {len(rest_df)} remaining rows to: {rest_csv_path}")

    # 7. Save the math heuristic predictions for submission
    math_submission_df = df_predictions[df_predictions["judge_source"] == "heuristic"].copy()
    math_submission_df = math_submission_df[["id", "predicted_label"]].rename(columns={"predicted_label": "label"})
    math_submission_path = "submission_math.csv"
    math_submission_df.to_csv(math_submission_path, index=False)
    print(f"[Main] Saved {len(math_submission_df)} math heuristic predictions to: {math_submission_path}")

    # --- Print Stats ---
    math_rows = df_predictions[df_predictions["judge_source"] != "non_math"]
    print(f"\n=== Running Statistics ===")
    print(f"Total inputs processed: {len(df_predictions)}")
    print(f"Math problems solved  : {len(math_rows)}")
    print(f"  Heuristic verdicts  : {(math_rows['judge_source'] == 'heuristic').sum()}")
    print(f"  LLM Judge verdicts  : {(math_rows['judge_source'] == 'llm').sum()}")
    print(f"  Execution failures  : {(math_rows['judge_source'] == 'exec_fail').sum()}")
    print(f"  Unresolved rows     : {(math_rows['judge_source'] == 'unresolved').sum()}")
    print("\n--- First 20 Rows Preview ---")
    print(df_predictions[["id", "prompt_bn", "response_bn", "calculated_answer", "predicted_label", "judge_source"]].head(20).to_string())


In [ ]:
NEW_test_csv= "rest_test_set.csv"

<h2> Part1.9 Yeet memory</h2>

In [ ]:
# ==============================================================
# GPU MEMORY CLEANUP: Release vLLM engine, Ray, and PyTorch Cache
# ==============================================================
import gc
import torch

print("Initial GPU Memory Usage:")
print(f"  Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
print(f"  Reserved:  {torch.cuda.memory_reserved() / (1024**3):.2f} GB")

# 1. Destroy the vLLM model parallel execution state
try:
    from vllm.model_executor.parallel_utils.parallel_state import destroy_model_parallel
except ImportError:
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel
    except ImportError:
        destroy_model_parallel = None

if destroy_model_parallel is not None:
    print("Destroying vLLM model parallel context...")
    try:
        destroy_model_parallel()
    except Exception as e:
        print(f"Error destroying model parallel context: {e}")

# 2. Shut down the Ray cluster backend (used by vLLM for orchestrating GPU execution)
try:
    import ray
    if ray.is_initialized():
        print("Shutting down active Ray cluster...")
        ray.shutdown()
except Exception as e:
    print(f"Error shutting down Ray: {e}")

# 3. Delete heavy model/data references from Python's global namespace
heavy_vars = ['llm', 'tokenizer', 'df_predictions', 'df']
for var in heavy_vars:
    if var in globals():
        print(f"Deleting python reference: {var}")
        del globals()[var]

# 4. Trigger garbage collector and clear PyTorch allocator caching
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # Releases shared memory references

print("\nFinal GPU Memory Usage (after cleanup):")
print(f"  Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
print(f"  Reserved:  {torch.cuda.memory_reserved() / (1024**3):.2f} GB")

# 5. Check actual system-wide GPU memory usage
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, text=True)
    print("\n--- System GPU Status ---")
    # Print the lines of nvidia-smi output containing GPU memory info
    lines = result.stdout.split('\n')
    for line in lines[:15]:
        print(line)
except Exception:
    pass


<H1> PART 2: MAKE RETRIVAL </H1>

In [ ]:
!pip install -q FlagEmbedding faiss-gpu pandas tqdm scikit-learn matplotlib seaborn


In [ ]:
import os
# Allow PyTorch to see all available GPUs for DataParallel
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import sys
import glob
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import torch
import faiss
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
# RAG Configuration and Flags
RETRIEVE_WITH_CONTEXT = True  # If False, retrieve ONLY when row context is empty. If True, retrieve for all.
RETRIEVE_TOP_K = 5
CANDIDATES_TO_RERANK = 40

# Search paths for database index and metadata
INDEX_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-final-but-without-bm25/bangla_knowledge.index"
if not os.path.exists(INDEX_PATH):
    INDEX_PATH = "../bangla_knowledge.index"
if not os.path.exists(INDEX_PATH):
    # Try Kaggle dataset path pattern
    kaggle_paths = glob.glob("/kaggle/input/**/bangla_knowledge.index", recursive=True)
    if kaggle_paths:
        INDEX_PATH = kaggle_paths[0]

METADATA_PATH = "/kaggle/input/datasets/barnobarno/rag-corpus-final-but-without-bm25/chunks_metadata.pkl"
if not os.path.exists(METADATA_PATH):
    METADATA_PATH = "../chunks_metadata.pkl"
if not os.path.exists(METADATA_PATH):
    # Try Kaggle dataset path pattern
    kaggle_paths = glob.glob("/kaggle/input/**/chunks_metadata.pkl", recursive=True)
    if kaggle_paths:
        METADATA_PATH = kaggle_paths[0]

print(f"RAG DB Paths configured:\nIndex: {INDEX_PATH}\nMetadata: {METADATA_PATH}")

In [ ]:
# Helper Functions for Prompt Formatting
def format_context(raw_context):
    if raw_context is None:
        return "(none)"
    if isinstance(raw_context, float) and pd.isna(raw_context):
        return "(none)"
    text = str(raw_context).strip()
    if text == "" or text.upper() == "[NULL]" or text.lower() == "nan":
        return "(none)"
    return text

INSTRUCTIONS = (
    "You are verifying Bengali Question-Answer pairs for factual correctness and faithfulness.\n"
    "You may be given a Context passage and/or Relevant Info. Follow these rules strictly:\n"
    "1. If a Context is provided, judge the Answer ONLY against that Context -- exact numbers, dates, and "
    "names must match. Do not use outside knowledge or Relevant Info to override or 'correct' the Context.\n"
    "2. If Context is '(none)', judge the Answer using the provided Relevant Info (if any) and general world knowledge.\n"
    "3. The Answer must be the correct TYPE of thing the Question asks for (e.g. if the Question asks "
    "'which year', a related entity name instead of a year is WRONG even if topically connected).\n"
    "4. Minor rewording or extra phrasing around a correct fact is fine; a wrong or contradicted fact is not.\n"
    "Respond with ONLY 'true' if the Answer is correct and faithful, or 'false' if it is incorrect, "
    "unfaithful, or contains hallucinations. Do not include any explanation or other text."
)

def build_prompt(context_text, question, answer, retrieved_context=""):
    prompt = f"{INSTRUCTIONS}\n\n"
    if retrieved_context and str(retrieved_context).strip() != "":
        prompt += f"Relevant Info: {retrieved_context}\n"
    prompt += (
        f"Context: {context_text}\n"
        f"Question: {question}\n"
        f"Answer: {answer}\n"
        "Output:"
    )
    return prompt

In [ ]:
# Load RAG assets (FAISS index, Metadata, BGE-M3 models)
# Since Tinker handles Qwen inference, local VRAM is empty. We will load models on GPU (cuda) if available!
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading RAG assets on device: {device}...")

if not os.path.exists(INDEX_PATH):
    print(f"Error: FAISS index file not found at '{INDEX_PATH}'. Please build it first.")
    sys.exit(1)
if not os.path.exists(METADATA_PATH):
    print(f"Error: Metadata pickle file not found at '{METADATA_PATH}'. Please build it first.")
    sys.exit(1)

# Load FAISS index
index = faiss.read_index(INDEX_PATH)
print(f"FAISS index loaded. Total indexed chunks: {index.ntotal}")

# Load metadata
with open(METADATA_PATH, "rb") as f:
    chunks = pickle.load(f)
print("Metadata chunks loaded successfully.")

# Load bi-encoder embedding model natively via transformers
print(f"Initializing BGE-M3 model natively on {device}...")
embed_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
embed_model = AutoModel.from_pretrained("BAAI/bge-m3").to(device)
if device == "cuda":
    embed_model = embed_model.half()
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for BGE-M3!")
        embed_model = torch.nn.DataParallel(embed_model)
embed_model.eval()

# Load cross-encoder reranker natively via transformers
print(f"Initializing BGE reranker natively on {device}...")
rerank_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-reranker-v2-m3")
reranker = AutoModelForSequenceClassification.from_pretrained("BAAI/bge-reranker-v2-m3").to(device)
if device == "cuda":
    reranker = reranker.half()
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for Reranker!")
        reranker = torch.nn.DataParallel(reranker)
reranker.eval()

print("All RAG models and database assets loaded successfully!")

In [ ]:
# Configure Paths and Load Dataset
INPUT_CSV = NEW_test_csv #= "rest_test_set.csv"  r"/kaggle/input/competitions/bengali-hallucination/test set.csv"
# if not os.path.exists(INPUT_CSV):
#     INPUT_CSV = "../EVALUATION NEWER.csv"
    
OUTPUT_CSV = "evaluation_results_qwen4B_rag.csv"
RETRIEVED_CSV = "evaluation_rag_queries_aro_FINAL.csv"
CONFUSION_MATRIX_PLOT = "qwen4b_rag_confusion_matrix.png"
METRICS_PLOT = "qwen4b_rag_evaluation_metrics.png"
CATEGORY_PLOT = "qwen4b_rag_category_metrics.png"

category = "all"

print(f"Loading evaluation set from: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)
print(f"Loaded evaluation set with {len(df)} rows.")

# if category.lower() != "all":
#     print(f"Filtering evaluation set for category: {category}")
#     df = df[df["category"].str.lower() == category.lower()].reset_index(drop=True)
#     if len(df) == 0:
#         available_cats = list(pd.read_csv(INPUT_CSV)["category"].dropna().unique())
#         print(f"Error: Category '{category}' not found. Available categories: {available_cats}")
#         exit(1)
#     print(f"Filtered evaluation set has {len(df)} rows.")

In [ ]:
# Step 1: Run RAG retrieval stage using GPU-accelerated batch operations
# We perform batch encoding and batch reranking to maximize GPU throughput and prevent terminal spam.
print("Running RAG retrieval stage (GPU-accelerated, batched)...")

# 1. Identify indices and questions that need RAG retrieval
retrieval_indices = []
retrieval_questions = []

for idx, row in df.iterrows():
    question = row.get("prompt_bn", "")
    context_val = format_context(row.get("context"))
    
    should_retrieve = False
    if context_val == "(none)":
        should_retrieve = True
    elif RETRIEVE_WITH_CONTEXT:
        should_retrieve = True
        
    if should_retrieve and pd.notna(question) and str(question).strip() != "":
        retrieval_indices.append(idx)
        retrieval_questions.append(str(question))

# 2. Run Batch Retrieval if there are questions to retrieve
evidence_results = {}
if retrieval_questions:
    print(f"Step 1.1: Batch encoding {len(retrieval_questions)} questions on {device}...")
    # Multi-GPU batching with tqdm
    batch_size = 256
    all_dense_vecs = []
    
    for i in tqdm(range(0, len(retrieval_questions), batch_size), desc="Encoding Queries"):
        batch_qs = retrieval_questions[i:i + batch_size]
        
        inputs = embed_tokenizer(batch_qs, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = embed_model(**inputs, return_dict=False)
            # Use [CLS] token for dense representation
            # When return_dict=False, outputs[0] is the last_hidden_state
            dense = outputs[0][:, 0]
            # L2 normalization for FAISS Inner Product
            dense = torch.nn.functional.normalize(dense, p=2, dim=1)
            all_dense_vecs.append(dense.cpu().numpy().astype("float32"))
            
    query_vecs = np.vstack(all_dense_vecs)
    
    # Normalize query vectors for cosine similarity (Inner Product)
    norms = np.linalg.norm(query_vecs, axis=1, keepdims=True)
    query_vecs = query_vecs / np.maximum(norms, 1e-12)
    
    print(f"Step 1.2: Searching FAISS index for top {CANDIDATES_TO_RERANK} candidates per query...")
    scores, indices = index.search(query_vecs, CANDIDATES_TO_RERANK)
    
    # Map index search results to metadata text chunks
    rerank_pairs = []
    query_candidates = []
    for i, question in enumerate(retrieval_questions):
        candidates_for_q = []
        for idx_val in indices[i]:
            if idx_val < len(chunks):
                candidates_for_q.append(chunks[idx_val]["text"])
                rerank_pairs.append([question, chunks[idx_val]["text"]])
        query_candidates.append(candidates_for_q)
        
    if rerank_pairs:
        print(f"Step 1.3: Batch reranking {len(rerank_pairs)} candidate pairs on {device}...")
        # Multi-GPU batching for reranker
        rerank_batch_size = 64
        rerank_scores = []
        for i in tqdm(range(0, len(rerank_pairs), rerank_batch_size), desc="Reranking"):
            batch_pairs = rerank_pairs[i:i + rerank_batch_size]
            inputs = rerank_tokenizer(batch_pairs, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                # return_dict=False makes it safer for DataParallel, outputs[0] are the logits
                scores = reranker(**inputs, return_dict=False)[0].view(-1).float()
                rerank_scores.extend(scores.cpu().numpy().tolist())
        
        # Sort and select top k evidence per question
        pair_idx = 0
        for i, question in enumerate(retrieval_questions):
            candidates_for_q = query_candidates[i]
            num_candidates = len(candidates_for_q)
            
            # Slice scores for this query
            q_scores = rerank_scores[pair_idx : pair_idx + num_candidates]
            pair_idx += num_candidates
            
            # Select top-k evidence
            ranked = sorted(zip(q_scores, candidates_for_q), reverse=True)[:RETRIEVE_TOP_K]
            
            evidence_parts = []
            for rank_idx, (score, text) in enumerate(ranked):
                evidence_parts.append(f"[তথ্যসূত্র {rank_idx+1}]: {text}")
                
            evidence_results[retrieval_indices[i]] = "\n".join(evidence_parts)

# 3. Populate retrieved context back to the DataFrame
retrieved_evidence_list = []
for idx, row in df.iterrows():
    if idx in evidence_results:
        retrieved_evidence_list.append(evidence_results[idx])
    else:
        retrieved_evidence_list.append("")

df["retrieved_context"] = retrieved_evidence_list

# Save intermediate results for inspection
df.to_csv(RETRIEVED_CSV, index=False)
print(f"Saved RAG queries and retrieved contexts to: {RETRIEVED_CSV}")

# Free up local RAG model memory to keep environment clean
del embed_model, reranker, index, chunks
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

<h1>PART 2.5 YEET MEMORY</h1>

In [ ]:
# ==============================================================
# GPU & SYSTEM RAM CLEANUP: Release FAISS, Bi-Encoder & Reranker
# ==============================================================
import gc
import torch

print("Initial GPU Memory Usage:")
if torch.cuda.is_available():
    print(f"  Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved() / (1024**3):.2f} GB")
else:
    print("  CUDA not available.")

# List of RAG models and index variables from Part 2
rag_vars = [
    'embed_model', 
    'reranker', 
    'index', 
    'chunks', 
    'embed_tokenizer', 
    'rerank_tokenizer',
    'df',
    'retrieved_evidence_list',
    'evidence_results'
]

# Delete variables from the global namespace
for var in rag_vars:
    if var in globals():
        print(f"Deleting python reference: {var}")
        del globals()[var]

# Run Garbage Collector to free CPU RAM
gc.collect()

# Clear PyTorch CUDA Cache to free VRAM
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # Release inter-process communication memory

print("\nFinal GPU Memory Usage (after cleanup):")
if torch.cuda.is_available():
    print(f"  Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved() / (1024**3):.2f} GB")
else:
    print("  CUDA not available.")

# Verify freed memory system-wide
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, text=True)
    print("\n--- System GPU Status ---")
    lines = result.stdout.split('\n')
    for line in lines[:15]:
         print(line)
except Exception:
    pass


<h1>PART 3: main model Qwen3.5 4B SFT </h1>

In [ ]:
NEW_TEST_CSV= RETRIEVED_CSV

In [ ]:
import subprocess
import sys

# 1. Clear out heavy frameworks that are known to cause dependency deadlocks
print("Uninstalling conflicting baseline packages...")
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y', 
    'keras', 'tensorflow', 'tensorboard'
], check=True)

# 2. Install vllm using the specific CUDA 12 wheel, along with its ecosystem
print("Installing vllm ecosystem (this may take a few minutes)...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install', 
    '--upgrade', 
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    '--extra-index-url', 'https://pypi.nvidia.com',
    # Bypass PyPI and use the official CUDA 12.9 wheel for vLLM 0.24.0
    'https://github.com/vllm-project/vllm/releases/download/v0.24.0/vllm-0.24.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl', 
    'unsloth', 
    'trl', 
    'openai_harmony'
], check=True)

print("Installation complete! You can now import your packages.")

In [ ]:
import os
import time
import pandas as pd

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
# ==============================================================================
# CONFIGURATION & FLAGS
# ==============================================================================
SMOKE_TEST = False            # Set to False for full inference
NUM_SMOKE_SAMPLES = 5        # Number of samples for smoke test
# MODEL_NAME = "cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit"
# MODEL_NAME = "cyankiwi/gemma-4-31B-it-AWQ-4bit"
MODEL_NAME = "/kaggle/input/datasets/nahidhossainredom/qwen3-5-4b-tinker-adapter-merged/merged_model"
INPUT_CSV = NEW_TEST_CSV #"/kaggle/input/rag-retrievals-ready-for-inference/evaluation_rag_queries_aro_BORO.csv"
OUTPUT_CSV2 = "submissionVLLM2.csv"
MAX_SEQ_LENGTH = 8192*2
MAX_OUTPUT_TOKENS = 7
SAFETY_MARGIN_TOKENS = 64
LOGPROBS_TOPK = 20            # top-k logprobs requested per token, used to find true/false probabilities


In [ ]:
import os
import glob
import re

print("Patching vLLM for Tesla T4 Shared Memory Limits...")

# 1. Locate the unified attention scripts inside the installed vLLM package
vllm_dir = "/usr/local/lib/python3.12/dist-packages/vllm"
matches = glob.glob(f"{vllm_dir}/**/triton_unified_attention*.py", recursive=True)

if not matches:
    print("Warning: Could not find the vLLM triton files to patch.")
else:
    for attn_file in matches:
        with open(attn_file, "r") as f:
            content = f.read()

        # 2. Force TILE_SIZE down to 16 (reduces shared memory footprint)
        content = re.sub(r'return 32', 'return 16', content)

        # 3. Disable Triton's deep pipelining by forcing num_stages=1 
        # This prevents the kernel from allocating huge double-buffers
        if 'num_stages=1' not in content:
            content = re.sub(r'(USE_FP8=output_scale is not None,?)', r'\1 num_stages=1,', content)

        with open(attn_file, "w") as f:
            f.write(content)

    # 4. Wipe the old compiled kernel caches so it rebuilds with our new settings
    os.system("rm -rf ~/.triton ~/.cache/vllm")
    print("Patch applied successfully! You can now import vLLM.")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} — {free/1e9:.1f}GB free / {total/1e9:.1f}GB total")

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
print(f"vLLM imported successfully!")

In [ ]:
# ==============================================================================
# LOAD MODEL & TOKENIZER
# ==============================================================================
print(f"\n{'='*60}")
print(f"Loading model: {MODEL_NAME}")
print(f"{'='*60}")
# Load model with T4 optimizations
llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=2,
    dtype="float16",
    # quantization="compressed-tensors",
    max_model_len=MAX_SEQ_LENGTH,
    gpu_memory_utilization=0.96,
    trust_remote_code=True,
    enforce_eager=True,
    enable_chunked_prefill=True,          
    max_num_batched_tokens=4096,
    max_num_seqs=16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)


In [ ]:
# ==============================================================================
# 2. PROMPT TEMPLATE & FEW-SHOT DATA
# ==============================================================================
# INSTRUCTIONS = (
#     "You are verifying Bengali Question-Answer pairs for factual correctness and faithfulness.\n"
#     "You may be given a Context passage and/or Relevant Info. Follow these rules strictly:\n"
#     "1. If a Context is provided, judge the Answer ONLY against that Context -- exact numbers, dates, and names must match.\n"
#     "2. If Context is '(none)', judge the Answer using the provided Relevant Info (if any) and general world knowledge.\n"
#     "3. The Answer must be the correct TYPE of thing the Question asks for (e.g. if the Question asks 'which year', a related entity name instead of a year is WRONG even if topically connected).\n"
#     "4. Minor rewording or extra phrasing around a correct fact is fine; a wrong or contradicted fact is not.\n"
#     "Respond with ONLY 'true' if the Answer is correct and faithful, or 'false' if it is incorrect, unfaithful, or contains hallucinations. Do not include any explanation or other text."
# )
INSTRUCTIONS = """
Your are a fact checker cheking answers to Question's by crosschecking with  
provided context and information. you must cross check if the response to the quesion is correct or incorrect ,
according to the context and provided information. Output "true" if the response is correct , "false" if the response is incorrect.
***IMPORTANT: only output either true or false.  
"""

FEW_SHOT_EXAMPLES = [
    # 1) Context present, answer fully supported by it -> true
    {
        "context": "উইন্ডোজে ইউনিকোড ভিত্তিক বাংলা লেখার জন্য ২০০৩ সালের ২৬শে মার্চ অভ্র কীবোর্ড সফটওয়্যারটি আবির্ভূত হয়। মেহদী হাসান খান নামে ময়মনসিংহ মেডিকেল কলেজের একজন ছাত্র ২০০৩ সালে অভ্র কীবোর্ড তৈরির কাজ শুরু করেন।",
        "question": "অভ্র কিবোর্ড কে উদ্ভাবন করেন?",
        "answer": "মেহদী হাসান খান",
        "output": "true",
    },
    # 2) Context present, answer contradicts a specific number in it -> false
    {
        "context": "পশ্চিমের মালভূমির সর্বোচ্চ অংশ অযোধ্যা পাহাড় (৬৭০ মিটার)। এছাড়া দক্ষিণে দলমা পাহাড় (৩৬৬ মিটার) ও উত্তর-পূর্বের পাঞ্চেত পাহাড়ও উল্লেখযোগ্য পাহাড়।",
        "question": "পুরুলিয়া জেলার অযোধ্যা পাহাড়ের উচ্চতা কত?",
        "answer": "৭২০ মিটার",
        "output": "false",
    },
    # 3) Context present, idiom interpreted literally instead of figuratively -> false
    {
        "context": "পুরো বংশে ওই ছেলেটাই তো শিবরাত্রির সলতে, তাকেও যদি যুদ্ধে পাঠিয়ে দাও, তবে আর বংশের বাতি জ্বালানোর কেউ থাকবে না।",
        "question": "এখানে 'শিবরাত্রির সলতে' অর্থ কী?",
        "answer": "পবিত্র প্রদীপ",
        "output": "false",
    },
    # 4) No context given, specific idiom with correct short meaning -> true
    {
        "context": None,
        "question": "'অগস্ত্য যাত্রা' বাগধারাটির অর্থ কী?",
        "answer": "চিরপ্রস্থান",
        "output": "true",
    },
    # 5) No context given, negative idiom answered with a positive antonym -> false
    {
        "context": None,
        "question": "'উড়নচণ্ডী' বাগধারাটির অর্থ কী?",
        "answer": "মিতব্যয়ী",
        "output": "false",
    },
]




def format_context(raw_context):
    if pd.isna(raw_context): return "(none)"
    text = str(raw_context).strip()
    return text if text and text.lower() != "nan" and text != "[NULL]" else "(none)"

# def build_prompt(context_text, question, answer, retrieved_context=""):
#     prompt = f"{INSTRUCTIONS}\n\n"
#     if retrieved_context and str(retrieved_context).strip() != "":
#         prompt += f"Relevant Info: {retrieved_context}\n"
#     prompt += (
#         f"Context: {context_text}\n"
#         f"Question: {question}\n"
#         f"Answer: {answer}\n"
#         "Output:"
#     )
#     return prompt


def build_prompt(context_text, question, answer, retrieved_context=""):
    prompt = f"{INSTRUCTIONS}\n\n"
    if retrieved_context and str(retrieved_context).strip() != "":
        prompt += f"Relevant Info: {retrieved_context}\n"
    prompt += (
        f"Context: {context_text}\n"
        "BASED ON THE PROVIDED INFO ,check the answer for the question below. ONLY output true of false."
        f"Question: {question}\n"
        f"Answer: {answer}\n"
        "Output:"
    )
    return prompt


In [ ]:
# ==============================================================================
# 2. SYSTEM INSTRUCTIONS & FEW-SHOT DATA (RAG OPTIMIZED)
# ==============================================================================
INSTRUCTIONS = """
Your are a fact checker cheking answers to Question's by crosschecking with  
provided context and information. you must cross check if the response to the quesion is correct or incorrect ,
according to the context and provided information. Output "true" if the response is correct , "false" if the response is incorrect.
***IMPORTANT: only output either true or false.  
"""

def format_context(raw_text):
    if pd.isna(raw_text): return "(none)"
    text = str(raw_text).strip()
    return text if text and text.lower() != "nan" and text != "[null]" else "(none)"

# Build the conversation history with the proper SYSTEM role
few_shot_messages = [
    {"role": "system", "content": INSTRUCTIONS}
]

for ex in FEW_SHOT_EXAMPLES:
    user_text = (
        f"Context: {format_context(ex.get('context'))}\n"
        f"Retrieved Context: {format_context(ex.get('retrieved_context'))}\n"
        f"Question: {ex['question']}\n"
        f"Answer: {ex['answer']}\n"
        "Output:"
    )
    few_shot_messages.append({"role": "user", "content": user_text})
    few_shot_messages.append({"role": "assistant", "content": ex['output']})

def build_user_query(context_text, retrieved_context, question, answer):
    return (
        f"Context: {context_text}\n"
        f"Retrieved Context: {retrieved_context}\n"
        "BASED ON THE PROVIDED INFO ,fact check the answer for the question below.only output true or false"
        f"Question: {question}\n"
        f"Answer: {answer}\n"
        "Output:"
    )

In [ ]:
# ==============================================================================
# TOKEN-BUDGET TRUNCATION FOR RETRIEVED CONTEXT
# ==============================================================================
import re

MIN_SNIPPET_TOKENS = 32  # always try to keep at least this many tokens of the
                          # top (most-similar) snippet, even if the budget is
                          # otherwise exhausted -- we never want zero retrieved
                          # context when a top candidate exists.


def split_snippets(retrieved_context_text):
    """Split the pre-formatted retrieved_context string into individual snippet
    blocks, delimited by a leading bracketed reference marker (e.g. '[তথ্যসূত্র N]:')
    at the start of a line. Falls back to treating the whole text as one snippet
    if no such markers are found (e.g. a single prose passage)."""
    if retrieved_context_text in ("(none)", "", None):
        return []
    parts = re.split(r"(?=^\[[^\]]+\]:)", retrieved_context_text, flags=re.MULTILINE)
    parts = [p.strip() for p in parts if p.strip()]
    return parts if parts else [retrieved_context_text]


def _hard_clip_to_budget(text, count_tokens_fn, max_tokens):
    """Binary-search clip `text` down to the largest prefix that fits in
    max_tokens. Returns '' if max_tokens <= 0."""
    if max_tokens <= 0 or not text:
        return ""
    lo, hi = 0, len(text)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if count_tokens_fn(text[:mid]) <= max_tokens:
            lo = mid
        else:
            hi = mid - 1
    return text[:lo]


def truncate_retrieved_context(retrieved_context_text, count_tokens_fn, max_tokens):
    """
    Greedily keep whole snippets, in original order (most-similar first), until
    the token budget is exhausted.

    Guarantee: as long as at least one snippet exists, we NEVER return empty
    context. If the full top snippet doesn't fit, we hard-clip it down to
    whatever fits -- reserving at least MIN_SNIPPET_TOKENS worth of budget for
    it even when max_tokens is otherwise ~0, since losing a fragment of the
    top (most relevant) candidate is far better than dropping retrieval
    entirely and silently falling back to only general world knowledge.

    Returns (truncated_text, was_truncated, kept_count, total_count).
    """
    if retrieved_context_text in ("(none)", "", None):
        return retrieved_context_text, False, 0, 0

    snippets = split_snippets(retrieved_context_text)
    total_count = len(snippets)

    # Always guarantee room for at least a fragment of the top snippet.
    effective_budget = max(max_tokens, MIN_SNIPPET_TOKENS)

    kept = []
    used_tokens = 0
    for i, snippet in enumerate(snippets):
        t = count_tokens_fn(snippet)
        if used_tokens + t > effective_budget:
            if i == 0:
                # Top snippet doesn't fully fit -- hard-clip it rather than
                # dropping it. This is the only place we ever end up with a
                # partial/empty result, and only when the top snippet itself
                # is bigger than the (possibly floor-raised) budget.
                remaining = effective_budget - used_tokens
                clipped = _hard_clip_to_budget(snippet, count_tokens_fn, remaining)
                if clipped:
                    kept.append(clipped + ("...[truncated]" if len(clipped) < len(snippet) else ""))
                    used_tokens += count_tokens_fn(kept[-1])
            break
        kept.append(snippet)
        used_tokens += t

    was_truncated = (len(kept) < total_count) or (kept and kept[0] != snippets[0])
    return "\n".join(kept), was_truncated, len(kept), total_count


def count_tokens(text):
    """Real token count via the loaded tokenizer (not an estimate)."""
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def fit_retrieved_context_to_budget(context_text, question, response, retrieved_context_text,
                                     max_model_len, max_output_tokens, safety_margin):
    """
    Truncates retrieved_context_text to fit within the model's context window,
    accounting for everything else in the prompt (system prompt + few-shot messages,
    context, question, response, and the reserved output budget).

    Even in the worst case (fixed_tokens alone nearly fills the budget), this
    still tries to preserve a small fragment of the top retrieved snippet via
    MIN_SNIPPET_TOKENS in truncate_retrieved_context, rather than returning
    empty retrieved context.
    """
    budget_for_prompt = max_model_len - max_output_tokens - safety_margin

    # Tokens used by everything except retrieved_context itself.
    dummy_query = build_user_query(context_text, "(none)", question, response)
    dummy_messages = few_shot_messages + [{"role": "user", "content": dummy_query}]
    dummy_prompt_text = tokenizer.apply_chat_template(dummy_messages, add_generation_prompt=True, tokenize=False)
    
    fixed_tokens = count_tokens(dummy_prompt_text)
    remaining_budget = max(0, budget_for_prompt - fixed_tokens)

    return truncate_retrieved_context(retrieved_context_text, count_tokens, remaining_budget)


In [ ]:
# ==============================================================================
# 3. MAIN INFERENCE LOOP
# ==============================================================================
import numpy as np
from scipy.special import softmax


def extract_true_false_probs(output):
    """
    Given a single vLLM RequestOutput, look at the logprob dict for the FIRST
    generated token and pull out the logprobs assigned to any token whose
    decoded text (stripped + lowercased) equals 'true' or 'false'. Different
    tokenizers can split these into variants like 'True', ' true', 'false',
    etc., so we scan all top-k candidates for that position and keep the max
    logprob seen for each label. The two logprobs are then renormalized with
    softmax into a probability pair that sums to 1.

    Returns (prob_true, prob_false). If a label never appears in the top-k
    logprobs, its raw logprob is treated as a large negative number (i.e.
    ~0 probability) rather than crashing.
    """
    VERY_NEGATIVE = -1e9

    step_logprobs = output.outputs[0].logprobs
    if not step_logprobs:
        # No logprobs available at all (e.g. logprobs=None) -- can't estimate.
        return np.nan, np.nan

    first_token_logprobs = step_logprobs[0]  # dict-like: token_id -> Logprob

    best_true = VERY_NEGATIVE
    best_false = VERY_NEGATIVE
    for lp in first_token_logprobs.values():
        decoded = lp.decoded_token.strip().lower()
        if decoded == "true":
            best_true = max(best_true, lp.logprob)
        elif decoded == "false":
            best_false = max(best_false, lp.logprob)

    prob_true, prob_false = softmax([best_true, best_false])
    return prob_true, prob_false


def force_fit_prompt(ctx, rc_fitted, question, response, max_model_len, max_output_tokens):
    """
    Builds the final prompt and GUARANTEES it fits the model's context window.
    Never skips a row. If the prompt (with the already-truncated retrieved
    context) still doesn't fit -- e.g. because chat-template overhead or the
    original Context field itself is unexpectedly large -- progressively:
      1. drops the retrieved context entirely (keeps Context + few-shot),
      2. then hard-clips the Context field itself,
    re-measuring the real tokenized prompt at each step. This always returns
    something that fits, so every row gets an answer.

    Returns (prompt_text, degradation_notes: list[str]).
    """
    budget = max_model_len - max_output_tokens
    notes = []

    def build(ctx_text, rc_text):
        query_text = build_user_query(ctx_text, rc_text, question, response)
        messages = few_shot_messages + [{"role": "user", "content": query_text}]
        return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False,enable_thinking=False)

    # Attempt 1: as given (context + already-budget-fitted retrieved context).
    prompt_text = build(ctx, rc_fitted)
    if count_tokens(prompt_text) <= budget:
        return prompt_text, notes

    # Attempt 2: drop retrieved context entirely, keep original Context.
    notes.append("dropped_retrieved_context")
    prompt_text = build(ctx, "(none)")
    if count_tokens(prompt_text) <= budget:
        return prompt_text, notes

    # Attempt 3: hard-clip the Context field too (rare: only if Context itself
    # plus system/few-shot prompt overhead exceeds the whole budget).
    notes.append("hard_clipped_context")
    # Binary-search clip ctx down to whatever fits alongside everything else.
    lo, hi = 0, len(ctx) if ctx else 0
    best = ""
    while lo <= hi:
        mid = (lo + hi) // 2
        candidate = (ctx[:mid] + "...[truncated]") if mid < len(ctx) else ctx[:mid]
        if count_tokens(build(candidate, "(none)")) <= budget:
            best = candidate
            lo = mid + 1
        else:
            hi = mid - 1
    prompt_text = build(best, "(none)")
    return prompt_text, notes


def main():
    # logprobs=LOGPROBS_TOPK asks vLLM to return the top-k logprobs for each
    # generated token, which we need in order to recover the true/false
    # probabilities below (not just the greedily-decoded text).
    sampling_params = SamplingParams(
        max_tokens=MAX_OUTPUT_TOKENS,
        temperature=0.0,
        logprobs=LOGPROBS_TOPK,
    )

    print(f"\n--- Loading Data: {INPUT_CSV} ---")
    df = pd.read_csv(INPUT_CSV)

    if SMOKE_TEST:
        print(f"!!! SMOKE TEST ENABLED: Running on {NUM_SMOKE_SAMPLES} samples !!!")
        df = df.head(NUM_SMOKE_SAMPLES)

    all_prompts = []
    all_ids = []
    all_rows_meta = []
    truncated_count = 0
    force_fit_count = 0

    print("\n--- Building Batched Prompts (every row is guaranteed a prompt -- none are skipped) ---")
    for _, row in df.iterrows():
        row_id = row.get("id")
        ctx = format_context(row.get("context"))
        rc_raw = format_context(row.get("retrieved_context"))
        question = row.get("prompt_bn", "")
        response = row.get("response_bn", "")
        
        rc_fitted, was_truncated, kept_n, total_n = fit_retrieved_context_to_budget(
            ctx, question, response, rc_raw,
            MAX_SEQ_LENGTH, MAX_OUTPUT_TOKENS, SAFETY_MARGIN_TOKENS,
        )
        if was_truncated:
            truncated_count += 1

        prompt_text, degradation_notes = force_fit_prompt(
            ctx, rc_fitted, question, response, MAX_SEQ_LENGTH, MAX_OUTPUT_TOKENS
        )
        if degradation_notes:
            force_fit_count += 1
            print(f"NOTE id={row_id}: prompt still over budget after retrieved-context "
                  f"truncation, applied extra fallback steps: {degradation_notes}")

        all_prompts.append(prompt_text)
        all_ids.append(row_id)
        all_rows_meta.append({
            "question": question, "response": response,
            "context_truncated": was_truncated,
            "snippets_kept": kept_n, "snippets_total": total_n,
            "degradation_notes": degradation_notes,
        })

    print(f"Total prompts to process: {len(all_prompts)} (== {len(df)} input rows, none skipped)")
    if truncated_count:
        print(f"NOTE: {truncated_count} row(s) had retrieved_context truncated to fit the token budget.")
    if force_fit_count:
        print(f"NOTE: {force_fit_count} row(s) needed extra fallback trimming beyond retrieved-context truncation.")
    
    print("\n--- Starting vLLM Inference ---")
    start_time = time.time()
    outputs = llm.generate(all_prompts, sampling_params) if all_prompts else []
    elapsed = time.time() - start_time
    
    if all_prompts:
        print(f"Done! Elapsed: {elapsed:.1f}s ({len(all_prompts)/elapsed:.1f} prompts/sec)")

    # Parse Results
    results = []
    for row_id, output in zip(all_ids, outputs):
        text = output.outputs[0].text.strip().lower()
        label = 1 if "true" in text else 0
        prob_true, prob_false = extract_true_false_probs(output)
        results.append({
            "id": row_id,
            "label": label,
            "prob_true": prob_true,
            "prob_false": prob_false,
        })
        if SMOKE_TEST:
            print(f"ID: {row_id} | Output: {text} | Label: {label} | "
                  f"P(true)={prob_true:.4f} P(false)={prob_false:.4f}")

    submission_df = pd.DataFrame(results).sort_values(by="id")
    submission_df.to_csv(OUTPUT_CSV2, index=False)
    print(f"\nSaved {len(submission_df)} results to {OUTPUT_CSV2}")
    print(submission_df.head())

if __name__ == "__main__":
    main()


In [ ]:
OUTPUT_CSV2

In [ ]:
a=pd.read_csv(OUTPUT_CSV2)
a.head()

In [ ]:
import pandas as pd

# Load files
df_math = pd.read_csv("submission_math.csv")
df_non_math = pd.read_csv(OUTPUT_CSV2)

# Combine and sort by id
df_final = pd.concat([df_math, df_non_math]).sort_values("id")

# Keep only 'id' and 'label' columns to match the format of submission_updated6.csv
df_final = df_final[["id", "label"]]

# Save final output
df_final.to_csv("final_submission.csv", index=False)
